# 01 — Data Gathering
Pull raw data from **nfl_data_py** (seasonal stats, snap counts, rosters)
and the **Sleeper API** (player metadata, dynasty context).
All data is cached as parquet files in `data/raw/` and loaded into SQLite.

Each nfl_data_py fetch is checkpointed season-by-season — safe to interrupt and resume.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from src.fetchers.nfl_fetcher import (
    fetch_seasonal_stats,
    fetch_weekly_stats,
    fetch_snap_counts,
    fetch_rosters,
)
from src.fetchers.sleeper_fetcher import fetch_players, fetch_adp
from src.pipeline.cleaner import load_to_db

import pandas as pd
pd.set_option('display.max_columns', 50)

## 1. Seasonal Stats (nfl_data_py)
Season-level aggregates: PPR points, targets, rushing, passing, EPA, usage metrics.

In [3]:
seasonal = fetch_seasonal_stats(seasons=list(range(2006, 2025)))
print(f'Shape: {seasonal.shape}')
seasonal.head(3)

  [1/19] 2006 — fetching... 536 rows saved
  [2/19] 2007 — fetching... 559 rows saved
  [3/19] 2008 — fetching... 557 rows saved
  [4/19] 2009 — fetching... 563 rows saved
  [5/19] 2010 — fetching... 578 rows saved
  [6/19] 2011 — fetching... 584 rows saved
  [7/19] 2012 — fetching... 601 rows saved
  [8/19] 2013 — fetching... 587 rows saved
  [9/19] 2014 — fetching... 584 rows saved
  [10/19] 2015 — fetching... 591 rows saved
  [11/19] 2016 — loaded from checkpoint
  [12/19] 2017 — loaded from checkpoint
  [13/19] 2018 — loaded from checkpoint
  [14/19] 2019 — loaded from checkpoint
  [15/19] 2020 — loaded from checkpoint
  [16/19] 2021 — loaded from checkpoint
  [17/19] 2022 — fetching... 619 rows saved
  [18/19] 2023 — loaded from checkpoint
  [19/19] 2024 — fetching... 607 rows saved
Combined seasonal stats: 11247 rows saved to /home/ruppdj/claude_testing/fantasy_football/src/fetchers/../../data/raw/seasonal_stats.parquet
Shape: (11247, 58)


,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,...,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr_x,special_teams_tds,fantasy_points,fantasy_points_ppr,games,tgt_sh,ay_sh,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh
0,00-0000108,2006,REG,1,1,11.0,0,0.0,0.0,0.0,0,0,0.0,11.0,0.0,-0.702337,0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,0.44,0.44,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.004402
1,00-0000166,2006,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,...,80.0,80.0,10.0,13.341678,0,14.608754,1.101259,0.822939,2.227946,0.0,28.00,46.00,8,0.076596,0.039448,0.112994,0.146452,0.106171,0.222222,0.138889,0.148148,0.164197,0.129381,0.680851,0.088261
2,00-0000251,2006,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.0,0.0,60,171.0,3,1.0,1.0,17.0,...,-15.0,100.0,6.0,0.888208,0,1.903175,1.160498,-0.188442,1.608837,0.0,41.60,62.60,14,0.046053,-0.004905,0.083126,0.065155,0.031835,0.000000,0.050000,0.045455,0.015918,0.025468,0.186404,0.071847


In [4]:
# Inspect available columns
print(seasonal.columns.tolist())

['player_id', 'season', 'season_type', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr_x', 'special_teams_tds', 'fantasy_points', 'fantasy_points_ppr', 'games', 'tgt_sh', 'ay_sh', 'yac_sh', 'wopr_y', 'ry_sh', 'rtd_sh', 'rfd_sh', 'rtdfd_sh', 'dom', 'w8dom', 'yptmpa', 'ppr_sh']


## 2. Snap Counts (offensive snap %)
Uses `nfl.import_snap_counts()` — the correct source for `offense_pct` (snap share).
Aggregated to season-level average per player.

In [7]:
snap_counts = fetch_snap_counts(seasons=list(range(2012, 2025)))
print(f'Snap counts shape: {snap_counts.shape}')
snap_counts.head(3)

  [1/13] 2012 — fetching... 0 player-season records saved
  [2/13] 2013 — fetching... 1594 player-season records saved
  [3/13] 2014 — fetching... 1622 player-season records saved
  [4/13] 2015 — fetching... 1632 player-season records saved
  [5/13] 2016 — loaded from checkpoint
  [6/13] 2017 — loaded from checkpoint
  [7/13] 2018 — loaded from checkpoint
  [8/13] 2019 — fetching... 1681 player-season records saved
  [9/13] 2020 — fetching... 1797 player-season records saved
  [10/13] 2021 — fetching... 1885 player-season records saved
  [11/13] 2022 — fetching... 1791 player-season records saved
  [12/13] 2023 — loaded from checkpoint
  [13/13] 2024 — fetching... 1780 player-season records saved
Combined snap counts: 20426 records saved to /home/ruppdj/claude_testing/fantasy_football/src/fetchers/../../data/raw/snap_counts.parquet
Snap counts shape: (20426, 3)


,player_id,season,avg_snap_pct
0,00-0000108,2013,0.0
1,00-0000585,2013,0.0
2,00-0004091,2013,0.0


## 3. Weekly Stats
Week-level data — useful for variance analysis and week-by-week EDA.

In [8]:
weekly = fetch_weekly_stats(seasons=list(range(2012, 2025)))
print(f'Weekly shape: {weekly.shape}')
weekly.head(3)

  [1/13] 2012 — fetching... Downcasting floats.
5150 rows saved
  [2/13] 2013 — fetching... Downcasting floats.
5022 rows saved
  [3/13] 2014 — fetching... Downcasting floats.
5129 rows saved
  [4/13] 2015 — fetching... Downcasting floats.
5101 rows saved
  [5/13] 2016 — fetching... Downcasting floats.
5062 rows saved
  [6/13] 2017 — fetching... Downcasting floats.
5107 rows saved
  [7/13] 2018 — fetching... Downcasting floats.
5070 rows saved
  [8/13] 2019 — fetching... Downcasting floats.
5046 rows saved
  [9/13] 2020 — fetching... Downcasting floats.
5198 rows saved
  [10/13] 2021 — fetching... Downcasting floats.
5452 rows saved
  [11/13] 2022 — fetching... Downcasting floats.
5391 rows saved
  [12/13] 2023 — loaded from checkpoint
  [13/13] 2024 — fetching... Downcasting floats.
5340 rows saved
Combined weekly stats: 67473 rows saved to /home/ruppdj/claude_testing/fantasy_football/src/fetchers/../../data/raw/weekly_stats.parquet
Weekly shape: (67473, 53)


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,...,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2012,12,REG,CLE,20,34,199.0,0,3.0,1.0,6.0,0,0,327.0,129.0,10.0,-8.355753,0,...,0.0,0,0.0,0.0,0.0,NaN,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,1.96,1.96
1,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2012,13,REG,BAL,25,36,276.0,1,1.0,2.0,6.0,1,0,336.0,128.0,12.0,6.709722,0,...,0.0,0,0.0,0.0,0.0,NaN,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,13.04,13.04
2,00-0004541,None,Donald Driver,WR,WR,None,GB,2012,2,REG,CHI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,...,0.0,0,0.0,0.0,0.0,NaN,0,1,1,26.0,1,0.0,0.0,24.0,2.0,1.0,2.862955,0,1.083333,0.03125,0.107143,0.121875,0.0,8.60,9.60


## 4. Rosters / Player ID Map
Static player metadata from `nfl.import_ids()`: birth_date, sleeper_id, pfr_id, college, etc.
Used as a lookup table joined to seasonal data on `player_id`.

In [9]:
rosters = fetch_rosters()
print(f'Rosters shape: {rosters.shape}')
rosters.head(3)

Loading rosters from cache: /home/ruppdj/claude_testing/fantasy_football/src/fetchers/../../data/raw/rosters.parquet
Rosters shape: (3217, 9)


,player_id,sleeper_id,pfr_id,name,position,birth_date,college,height,weight
0,00-0040676,12522.0,WardCa00,Cam Ward,QB,2002-05-25,Miami (FL),74.0,219.0
1,00-0040668,12524.0,SandSh00,Shedeur Sanders,QB,2002-02-07,Colorado,74.0,212.0
2,00-0040691,12508.0,DartJa00,Jaxson Dart,QB,2003-05-13,Ole Miss,74.0,223.0


## 5. Sleeper Player Metadata

In [10]:
sleeper_players = fetch_players()
print(f'Sleeper players shape: {sleeper_players.shape}')
sleeper_players.head(3)

Loading Sleeper players from cache: /home/ruppdj/claude_testing/fantasy_football/src/fetchers/../../data/raw/sleeper_players.parquet
Sleeper players shape: (4001, 18)


,sleeper_id,full_name,first_name,last_name,position,fantasy_positions,team,age,years_exp,birth_date,college,height,weight,status,injury_status,depth_chart_order,search_rank,number
0,6462,Ellis Richardson,Ellis,Richardson,TE,TE,None,26.0,3.0,1995-02-12,Georgia Southern,75,245,Active,None,NaN,9999999.0,45.0
1,7926,Carl Tucker,Carl,Tucker,TE,TE,None,24.0,1.0,1997-02-06,Alabama,74,250,Active,None,NaN,426.0,0.0
2,1408,Le'Veon Bell,Le'Veon,Bell,RB,RB,TB,29.0,9.0,1992-02-18,Michigan State,73,225,Active,None,NaN,235.0,6.0


In [11]:
# Position breakdown in Sleeper data
sleeper_players['position'].value_counts()

WR     1712
RB      898
TE      813
QB      463
FB      109
T         2
CB        2
ATH       1
SS        1
Name: position, dtype: int64

## 6. Load Raw Tables into SQLite

In [12]:
load_to_db(seasonal, 'nfl_seasonal_raw')
load_to_db(weekly, 'nfl_weekly_raw')
load_to_db(rosters, 'nfl_rosters')
load_to_db(sleeper_players, 'sleeper_players')
load_to_db(snap_counts, 'snap_counts')
print('All raw tables loaded to fantasy.db')

Loaded 11247 rows into table 'nfl_seasonal_raw'
Loaded 67473 rows into table 'nfl_weekly_raw'
Loaded 3217 rows into table 'nfl_rosters'
Loaded 4001 rows into table 'sleeper_players'
Loaded 20426 rows into table 'snap_counts'
All raw tables loaded to fantasy.db


## 7. Sanity Checks

In [13]:
# Seasons covered
print('Seasons:', sorted(seasonal['season'].unique()))

# Positions available
if 'position' in seasonal.columns:
    print('Positions:', seasonal['position'].value_counts().to_dict())

# Null rate in key columns
key_cols = ['fantasy_points_ppr', 'target_share', 'air_yards_share']
for col in key_cols:
    if col in seasonal.columns:
        null_pct = seasonal[col].isna().mean() * 100
        print(f'  {col}: {null_pct:.1f}% null')

# Snap counts coverage
print(f'\nSnap count seasons: {sorted(snap_counts["season"].unique())}')
print(f'Snap count avg_snap_pct range: {snap_counts["avg_snap_pct"].min():.2f} – {snap_counts["avg_snap_pct"].max():.2f}')

Seasons: [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
  fantasy_points_ppr: 0.0% null
  target_share: 0.0% null
  air_yards_share: 0.0% null

Snap count seasons: [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Snap count avg_snap_pct range: 0.00 – 1.00
